# 💼 LinkedIn AI Agent
**Full-featured LinkedIn automation with human-like behavior.**

### Features:
- 🔐 Login (session cookies saved)
- 📝 Create Text / Image / Video Posts
- 👍 Like, React to Posts
- 💬 Comment & Reply
- 🤝 Send Connection Requests
- 📩 Send Direct Messages
- 📊 View Post Analytics
- 🔍 Search People & Jobs
- ⏰ Post Scheduler

> ⚠️ Uses `undetected-chromedriver` + randomized delays to mimic real user sessions.

In [ ]:
# ============================================================
# CELL 1: Imports & Human Behavior Engine
# ============================================================
import time
import random
import json
import os
import pickle
from pathlib import Path
from datetime import datetime
from getpass import getpass

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

import colorama
from colorama import Fore, Style
colorama.init()

# ── Timing helpers ───────────────────────────────────────────
def human_delay(mn=1.5, mx=4.0):
    t = random.uniform(mn, mx)
    print(f"{Fore.CYAN}  ⏳ {t:.1f}s{Style.RESET_ALL}", end="\r")
    time.sleep(t)

def long_delay(mn=8.0, mx=22.0):
    t = random.uniform(mn, mx)
    print(f"{Fore.YELLOW}  😴 long pause {t:.1f}s...{Style.RESET_ALL}")
    time.sleep(t)

def scroll_pause(): time.sleep(random.uniform(0.4, 1.1))

def log(msg, level="info"):
    c = {"info": Fore.GREEN, "warn": Fore.YELLOW, "error": Fore.RED, "data": Fore.MAGENTA}.get(level, Fore.WHITE)
    print(f"{c}[{datetime.now():%H:%M:%S}] {msg}{Style.RESET_ALL}")

def human_type(element, text: str, clear_first: bool = True):
    """Type text character by character with random speed."""
    if clear_first:
        element.clear()
        time.sleep(random.uniform(0.3, 0.7))
    for char in text:
        element.send_keys(char)
        time.sleep(random.uniform(0.04, 0.13))
    time.sleep(random.uniform(0.3, 0.8))

def safe_click(driver, element):
    """Scroll to element then click."""
    driver.execute_script("arguments[0].scrollIntoView({block:'center'})", element)
    time.sleep(random.uniform(0.4, 0.9))
    ActionChains(driver).move_to_element(element).pause(random.uniform(0.2, 0.5)).click().perform()

log("✅ Imports & human engine loaded.")

In [ ]:
# ============================================================
# CELL 2: Browser Setup & Login
# ============================================================
USERNAME     = input("LinkedIn email: ")
PASSWORD     = getpass("LinkedIn password: ")
COOKIES_FILE = Path(f"li_cookies_{USERNAME.split('@')[0]}.pkl")

def create_driver():
    opts = uc.ChromeOptions()
    opts.add_argument("--start-maximized")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--no-sandbox")
    opts.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/122.0.0.0 Safari/537.36")
    driver = uc.Chrome(options=opts)
    return driver

def wait(driver, css_selector: str, timeout: int = 15):
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, css_selector))
    )

def linkedin_login(driver):
    driver.get("https://www.linkedin.com")
    human_delay(2, 4)

    # Try loading cookies
    if COOKIES_FILE.exists():
        log("Loading saved cookies...")
        for c in pickle.loads(COOKIES_FILE.read_bytes()):
            try: driver.add_cookie(c)
            except: pass
        driver.refresh()
        human_delay(3, 5)
        if "feed" in driver.current_url or "mynetwork" in driver.current_url:
            log("✅ Logged in via cookies.")
            return

    # Fresh login
    driver.get("https://www.linkedin.com/login")
    human_delay(2, 4)

    email_el = wait(driver, "#username")
    human_type(email_el, USERNAME)
    human_delay(0.5, 1.5)

    pass_el = driver.find_element(By.CSS_SELECTOR, "#password")
    human_type(pass_el, PASSWORD)
    human_delay(0.5, 1.5)

    driver.find_element(By.CSS_SELECTOR, "button[type='submit']").click()
    human_delay(4, 7)

    # Save cookies
    COOKIES_FILE.write_bytes(pickle.dumps(driver.get_cookies()))
    log("✅ Logged in & cookies saved.")

driver = create_driver()
linkedin_login(driver)
log(f"LinkedIn session active.", "data")

In [ ]:
# ============================================================
# CELL 3: Create a Text Post
# ============================================================
def create_text_post(text: str):
    log("Creating a text post...")
    driver.get("https://www.linkedin.com/feed/")
    human_delay(3, 6)

    # Click 'Start a post'
    try:
        start_btn = wait(driver, ".share-box-feed-entry__trigger")
        safe_click(driver, start_btn)
    except:
        start_btn = wait(driver, "[data-control-name='share.sharebox_trigger']")
        safe_click(driver, start_btn)

    human_delay(2, 4)

    # Type in the post editor
    editor = wait(driver, ".ql-editor, [data-placeholder='What do you want to talk about?']")
    safe_click(driver, editor)
    human_type(editor, text, clear_first=False)
    human_delay(1, 3)

    # Click Post button
    post_btn = wait(driver, ".share-actions__primary-action")
    safe_click(driver, post_btn)
    human_delay(3, 6)
    log("✅ Post published!", "data")
    long_delay(10, 20)

# ── Example ──────────────────────────────────────────────────
# create_text_post("Excited to share my latest project! 🚀\n\n#Python #AI #LinkedIn")

In [ ]:
# ============================================================
# CELL 4: Upload Image/Video Post
# ============================================================
def create_media_post(media_path: str, caption: str):
    """
    Post with an image or video attachment.
    media_path: absolute path to image/video file
    """
    log(f"Creating media post: {media_path}")
    driver.get("https://www.linkedin.com/feed/")
    human_delay(3, 6)

    start_btn = wait(driver, ".share-box-feed-entry__trigger")
    safe_click(driver, start_btn)
    human_delay(2, 4)

    # Click media upload button
    media_btn = wait(driver, "button[aria-label*='Add a photo']") if media_path.lower().endswith(('.jpg','.jpeg','.png')) \
        else wait(driver, "button[aria-label*='Add a video']")
    safe_click(driver, media_btn)
    human_delay(1, 3)

    # Send file to hidden input
    file_input = driver.find_element(By.CSS_SELECTOR, "input[type='file']")
    file_input.send_keys(os.path.abspath(media_path))
    human_delay(4, 8)

    # Write caption
    editor = wait(driver, ".ql-editor")
    safe_click(driver, editor)
    human_type(editor, caption, clear_first=False)
    human_delay(2, 4)

    post_btn = wait(driver, ".share-actions__primary-action")
    safe_click(driver, post_btn)
    human_delay(5, 10)
    log("✅ Media post published!", "data")
    long_delay()

# ── Example ──────────────────────────────────────────────────
# create_media_post(r"C:\path\to\image.jpg", "Check out this amazing graphic! 🎨")

In [ ]:
# ============================================================
# CELL 5: Like & React to Posts
# ============================================================
def like_feed_posts(count: int = 5):
    """Scroll through feed and like posts naturally."""
    log(f"Browsing feed to like up to {count} posts...")
    driver.get("https://www.linkedin.com/feed/")
    human_delay(3, 5)

    liked = 0
    scroll_attempts = 0
    while liked < count and scroll_attempts < 15:
        # Find like buttons that haven't been pressed
        like_buttons = driver.find_elements(By.CSS_SELECTOR, 
            "button.react-button__trigger[aria-pressed='false']")
        
        for btn in like_buttons:
            if liked >= count:
                break
            if random.random() < 0.6:   # 60% chance to like
                try:
                    safe_click(driver, btn)
                    liked += 1
                    log(f"  👍 Liked post #{liked}")
                    human_delay(2, 5)
                except Exception as e:
                    pass
            else:
                scroll_pause()

        # Scroll to load more
        driver.execute_script("window.scrollBy(0, random.randint(400, 800))")
        scroll_pause()
        scroll_attempts += 1

    log(f"✅ Liked {liked} posts.", "data")

def like_post_by_url(post_url: str):
    """Navigate to specific post and like it."""
    driver.get(post_url)
    human_delay(3, 5)
    btn = wait(driver, "button.react-button__trigger")
    safe_click(driver, btn)
    log("✅ Post liked.")
    human_delay(2, 5)

# ── Example ──────────────────────────────────────────────────
# like_feed_posts(count=4)

In [ ]:
# ============================================================
# CELL 6: Comment & Reply
# ============================================================
def comment_on_post(post_url: str, comment_text: str):
    """Navigate to post and leave a comment."""
    log(f"Commenting on {post_url}")
    driver.get(post_url)
    human_delay(3, 6)

    # Click 'Comment' button
    comment_btn = wait(driver, "button[aria-label*='comment']") 
    safe_click(driver, comment_btn)
    human_delay(1, 3)

    # Type comment
    editor = wait(driver, ".comments-comment-box__form .ql-editor")
    safe_click(driver, editor)
    human_type(editor, comment_text, clear_first=False)
    human_delay(1, 2)

    # Submit
    submit = wait(driver, "button.comments-comment-box__submit-button")
    safe_click(driver, submit)
    human_delay(2, 5)
    log(f"✅ Comment posted: '{comment_text}'", "data")
    long_delay(8, 18)

def reply_to_comment(post_url: str, commenter_name: str, reply_text: str):
    """Reply to the first comment by a specific person."""
    log(f"Replying to {commenter_name}'s comment...")
    driver.get(post_url)
    human_delay(3, 5)

    # Find reply button near commenter
    comments = driver.find_elements(By.CSS_SELECTOR, ".comments-comment-item")
    for c in comments:
        try:
            name = c.find_element(By.CSS_SELECTOR, ".comments-post-meta__name").text
            if commenter_name.lower() in name.lower():
                reply_btn = c.find_element(By.CSS_SELECTOR, "button.comments-comment-item__reply-action-toggle")
                safe_click(driver, reply_btn)
                human_delay(1, 3)
                editor = wait(driver, ".comments-comment-box .ql-editor")
                human_type(editor, reply_text, clear_first=False)
                submit = wait(driver, "button.comments-comment-box__submit-button")
                safe_click(driver, submit)
                log(f"✅ Replied to {commenter_name}.", "data")
                long_delay()
                return
        except: continue
    log(f"Comment by {commenter_name} not found.", "warn")

# ── Example ──────────────────────────────────────────────────
# comment_on_post("https://www.linkedin.com/posts/USERNAME_POST_ID", "Great insights! 🙌")

In [ ]:
# ============================================================
# CELL 7: Send Connection Request
# ============================================================
def send_connection_request(profile_url: str, note: str = ""):
    """
    Send a connection request to a LinkedIn profile.
    profile_url: e.g., 'https://www.linkedin.com/in/some-person'
    note: optional personalized note (max 300 chars)
    """
    log(f"Visiting {profile_url}")
    driver.get(profile_url)
    human_delay(4, 8)

    try:
        # Find 'Connect' button
        connect_btn = wait(driver, "button[aria-label*='Connect']")
        safe_click(driver, connect_btn)
        human_delay(1, 3)

        if note:
            add_note = wait(driver, "button[aria-label*='Add a note']")
            safe_click(driver, add_note)
            human_delay(1, 2)
            note_box = wait(driver, "#custom-message")
            human_type(note_box, note[:300])
            human_delay(1, 2)

        send = wait(driver, "button[aria-label*='Send now'], button[aria-label*='Send invitation']")
        safe_click(driver, send)
        human_delay(2, 5)
        log(f"✅ Connection request sent!", "data")
    except Exception as e:
        log(f"Could not send connection: {e}", "warn")

    long_delay(10, 25)

# ── Example ──────────────────────────────────────────────────
# send_connection_request(
#     "https://www.linkedin.com/in/someone",
#     note="Hi! I loved your post on AI. Let's connect!"
# )

In [ ]:
# ============================================================
# CELL 8: Send Direct Messages
# ============================================================
def send_dm(profile_url: str, message: str):
    """Send a direct message to a connection."""
    log(f"Sending DM via {profile_url}")
    driver.get(profile_url)
    human_delay(3, 6)

    try:
        msg_btn = wait(driver, "button[aria-label*='Message']")
        safe_click(driver, msg_btn)
        human_delay(2, 4)

        msg_box = wait(driver, ".msg-form__contenteditable")
        safe_click(driver, msg_box)
        human_type(msg_box, message, clear_first=False)
        human_delay(1, 3)

        send = wait(driver, "button.msg-form__send-button")
        safe_click(driver, send)
        human_delay(2, 5)
        log("✅ DM sent!", "data")
    except Exception as e:
        log(f"DM failed: {e}", "warn")

    long_delay(10, 25)

def get_messages():
    """Open LinkedIn messages inbox."""
    driver.get("https://www.linkedin.com/messaging/")
    human_delay(3, 5)
    log("📩 Messages inbox opened. Check browser.", "data")

# ── Example ──────────────────────────────────────────────────
# send_dm("https://www.linkedin.com/in/someone", "Hey! Loved your recent article 📖")

In [ ]:
# ============================================================
# CELL 9: Search People & Jobs
# ============================================================
def search_people(query: str, count: int = 5):
    """Search for LinkedIn profiles."""
    log(f"Searching people: {query}")
    url = f"https://www.linkedin.com/search/results/people/?keywords={query.replace(' ', '%20')}"
    driver.get(url)
    human_delay(3, 6)

    results = driver.find_elements(By.CSS_SELECTOR, ".entity-result__title-text a")[:count]
    people = []
    print(f"\n👥 People results for '{query}':")
    for r in results:
        print(f"  • {r.text.strip()} → {r.get_attribute('href')}")
        people.append({"name": r.text.strip(), "url": r.get_attribute("href")})
        scroll_pause()
    return people

def search_jobs(keyword: str, location: str = ""):
    """Search LinkedIn Jobs."""
    log(f"Searching jobs: {keyword} in {location or 'anywhere'}")
    url = (f"https://www.linkedin.com/jobs/search/?"
           f"keywords={keyword.replace(' ', '%20')}&location={location.replace(' ', '%20')}")
    driver.get(url)
    human_delay(3, 6)

    jobs = driver.find_elements(By.CSS_SELECTOR, ".job-card-container")
    print(f"\n💼 Found {len(jobs)} jobs for '{keyword}':")
    for j in jobs[:5]:
        try:
            title   = j.find_element(By.CSS_SELECTOR, ".job-card-list__title").text
            company = j.find_element(By.CSS_SELECTOR, ".job-card-container__company-name").text
            print(f"  • {title} @ {company}")
        except: pass
        scroll_pause()

# ── Example ──────────────────────────────────────────────────
# search_people("Python Developer")
# search_jobs("Machine Learning Engineer", location="Remote")

In [ ]:
# ============================================================
# CELL 10: Post Analytics
# ============================================================
def get_my_posts_analytics():
    """Check analytics for your recent posts from My Activity."""
    log("Fetching post analytics...")
    driver.get("https://www.linkedin.com/in/me/recent-activity/shares/")
    human_delay(4, 7)

    posts = driver.find_elements(By.CSS_SELECTOR, ".feed-shared-update-v2")
    print(f"\n📊 Your Recent Posts Analytics:")
    for i, p in enumerate(posts[:5]):
        try:
            # Impressions / reactions
            reactions = p.find_element(By.CSS_SELECTOR, ".social-details-social-counts__reactions").text
            comments  = p.find_element(By.CSS_SELECTOR, ".social-details-social-counts__comments").text
            print(f"  Post {i+1}: 👍 {reactions}  💬 {comments}")
        except:
            print(f"  Post {i+1}: (analytics not visible here)")
        scroll_pause()

    log("Tip: Full analytics available at LinkedIn Creator Analytics page.", "warn")

# ── Example ──────────────────────────────────────────────────
# get_my_posts_analytics()

In [ ]:
# ============================================================
# CELL 11: Post Scheduler
# ============================================================
import schedule as sched
import threading

def schedule_post(text: str, hour: int, minute: int, media_path: str = None):
    time_str = f"{hour:02d}:{minute:02d}"
    def job():
        log(f"⏰ Scheduled post running at {time_str}")
        if media_path:
            create_media_post(media_path, text)
        else:
            create_text_post(text)
    sched.every().day.at(time_str).do(job)
    log(f"📅 Post scheduled for {time_str} daily.")

def run_scheduler():
    def _loop():
        while True:
            sched.run_pending()
            time.sleep(30)
    threading.Thread(target=_loop, daemon=True).start()
    log("✅ Scheduler running in background.")

# ── Example ──────────────────────────────────────────────────
# schedule_post("Good morning LinkedIn! 🌅 #Motivation", hour=8, minute=30)
# run_scheduler()

In [ ]:
# ============================================================
# CELL 12: Cleanup
# ============================================================
def close_browser():
    log("Closing browser...")
    driver.quit()
    log("✅ Browser closed.")

# Run this when you're done:
# close_browser()